# OCR Model Comparison

Compare transcription quality across four vision-language OCR models on the
same column strips produced by the `annotate_page_structured()` pipeline.

| Model | Size | Provider |
|---|---|---|
| Gemini 2.5 Flash | — | Google API |
| DeepSeek-OCR-2 | 3 B | HF Inference Endpoint |
| LightOnOCR-2 | 1 B | HF Inference Endpoint |
| GLM-OCR | 0.9 B | HF Inference Endpoint |

## Prerequisites

1. Run the structured pipeline at least once so column strips exist in `data/interim/strips/`.
2. Set API keys in `.env` (see `.env.example`):
   - `GEMINI_API_KEY`
   - `HF_TOKEN`
   - `DEEPSEEK_OCR_ENDPOINT`, `LIGHTON_OCR_ENDPOINT`, `GLM_OCR_ENDPOINT`
3. Deploy HuggingFace Inference Endpoints for each model at https://ui.endpoints.huggingface.co/

In [ ]:
from __future__ import annotations
import json, sys, time, textwrap
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, HTML

# ── repo root ──────────────────────────────────────────────────────────────
REPO = Path().resolve().parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

PROCESSED = REPO / "data" / "processed"
STRIPS_DIR = REPO / "data" / "interim" / "strips"

print("Repo root:", REPO)

## 1 · Initialise OCR backends

Only backends whose API keys / endpoint URLs are configured will be loaded.
Missing backends are skipped with a warning.

In [ ]:
from newspapers.ocr import get_all_backends

backends = get_all_backends(skip_missing=True)

if not backends:
    raise RuntimeError(
        "No OCR backends configured. "
        "Set GEMINI_API_KEY and/or HF_TOKEN + endpoint URLs in .env"
    )

print(f"{len(backends)} backend(s) available:")
for b in backends:
    print(f"  - {b.name}")

## 2 · Select page & run column segmentation

In [ ]:
from newspapers.segmentation.structure import analyse_page_structure

# Pick the first processed image (change index to try a different page)
jpgs = sorted(PROCESSED.glob("*.jpg"))
if not jpgs:
    raise FileNotFoundError(f"No .jpg files in {PROCESSED}")

IMAGE_PATH = jpgs[0]
print("Selected page:", IMAGE_PATH.name)

# Use high-res PNG if available
struct_img = IMAGE_PATH.with_suffix(".png") if IMAGE_PATH.with_suffix(".png").exists() else IMAGE_PATH

column_bounds, strips, profile, skew_angle = analyse_page_structure(
    struct_img, n_columns_hint=8
)
print(f"Skew: {skew_angle:.2f}°  |  {len(column_bounds)} boundaries  |  {len(strips)} strips")
print("Strips:", [s.strip_id for s in strips])

## 3 · Preview strips

Quick visual of all column strips before running OCR.

In [ ]:
# Filter to column strips only (skip full-page thumbnail)
col_strips = [s for s in strips if s.strip_id not in ("full",)]

n = len(col_strips)
cols_per_row = min(5, n)
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(cols_per_row * 3, rows * 8))
axes = np.array(axes).flatten()

for i, strip in enumerate(col_strips):
    img = Image.open(strip.image_path)
    axes[i].imshow(img)
    axes[i].set_title(f"{strip.strip_id}\n{img.width}×{img.height}", fontsize=8, pad=3)
    axes[i].axis("off")

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(f"Column strips — {IMAGE_PATH.stem}", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 4 · Run OCR on selected strips

Pick a subset of strips to transcribe (all columns can be slow with 4 models).
Results are cached to a JSON file so re-runs are fast.

In [ ]:
# Choose which strips to compare (edit this list as needed)
# By default: masthead + first 3 columns
STRIP_IDS_TO_COMPARE = ["masthead", "col_1", "col_2", "col_3"]

strips_to_run = [s for s in strips if s.strip_id in STRIP_IDS_TO_COMPARE]
print(f"Will transcribe {len(strips_to_run)} strips with {len(backends)} backends ")
print(f"= {len(strips_to_run) * len(backends)} total API calls")

In [ ]:
# Cache file to avoid redundant API calls on re-runs
CACHE_PATH = REPO / "data" / "interim" / f"ocr_comparison_{IMAGE_PATH.stem}.json"

# Load existing cache
if CACHE_PATH.exists():
    cached_results = json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {sum(len(v) for v in cached_results.values())} cached transcriptions")
else:
    cached_results = {}

results: dict[str, dict[str, str]] = defaultdict(dict)
results.update({k: dict(v) for k, v in cached_results.items()})

for strip in strips_to_run:
    img = Image.open(strip.image_path).convert("RGB")
    for backend in backends:
        key = strip.strip_id
        if key in results and backend.name in results[key]:
            print(f"  [cached] {strip.strip_id} / {backend.name}")
            continue
        print(f"  Transcribing {strip.strip_id} with {backend.name}...", end=" ", flush=True)
        t0 = time.time()
        try:
            text = backend.transcribe(img)
            results[key][backend.name] = text
            elapsed = time.time() - t0
            print(f"{len(text)} chars in {elapsed:.1f}s")
        except Exception as exc:
            results[key][backend.name] = f"[ERROR] {exc}"
            print(f"FAILED: {exc}")

# Save cache
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
CACHE_PATH.write_text(json.dumps(dict(results), indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nResults cached to {CACHE_PATH.relative_to(REPO)}")

## 5 · Side-by-side comparison

For each strip: the image on the left, transcriptions from each model in columns.

In [ ]:
import html as html_mod

def render_comparison_html(strip_id: str, strip_image_path: Path, transcriptions: dict[str, str]) -> str:
    """Build an HTML table comparing transcriptions for one strip."""
    model_names = list(transcriptions.keys())
    n_models = len(model_names)

    header = "".join(f"<th style='text-align:left; padding:6px; border:1px solid #ddd;'>{html_mod.escape(m)}</th>" for m in model_names)
    cells = "".join(
        f"<td style='vertical-align:top; padding:6px; border:1px solid #ddd; "
        f"white-space:pre-wrap; font-family:monospace; font-size:11px; max-width:300px;'>"
        f"{html_mod.escape(transcriptions[m])}</td>"
        for m in model_names
    )

    return f"""
    <h3 style="margin-top:24px;">{html_mod.escape(strip_id)}</h3>
    <table style="border-collapse:collapse; width:100%;">
      <tr>{header}</tr>
      <tr>{cells}</tr>
    </table>
    """

html_parts = ["<h2>Transcription Comparison</h2>"]
for strip in strips_to_run:
    sid = strip.strip_id
    if sid not in results:
        continue
    # Show the strip image
    html_parts.append(render_comparison_html(sid, strip.image_path, results[sid]))

display(HTML("".join(html_parts)))

## 6 · Strip image + transcription (visual)

Matplotlib version: strip image on the left, transcription text panels on the right.

In [ ]:
for strip in strips_to_run:
    sid = strip.strip_id
    if sid not in results:
        continue

    transcriptions = results[sid]
    model_names = list(transcriptions.keys())
    n_models = len(model_names)

    fig, axes = plt.subplots(1, n_models + 1, figsize=(4 * (n_models + 1), 10),
                              gridspec_kw={"width_ratios": [1.5] + [1] * n_models})

    # Strip image
    img = Image.open(strip.image_path)
    axes[0].imshow(img)
    axes[0].set_title(f"{sid}\n{img.width}×{img.height}", fontsize=9)
    axes[0].axis("off")

    # Transcription panels
    for j, model_name in enumerate(model_names):
        ax = axes[j + 1]
        ax.axis("off")
        text = transcriptions[model_name]
        # Truncate long text for display
        display_text = text[:1500] + ("\n[...truncated]" if len(text) > 1500 else "")
        ax.text(0.02, 0.98, display_text, transform=ax.transAxes,
                fontsize=6, verticalalignment="top", fontfamily="monospace",
                wrap=True)
        n_chars = len(text)
        n_words = len(text.split())
        ax.set_title(f"{model_name}\n{n_chars} chars / {n_words} words", fontsize=8)

    plt.suptitle(f"OCR Comparison — {sid}", fontsize=12)
    plt.tight_layout()
    plt.show()

## 7 · Summary statistics

In [ ]:
rows = []
for sid, transcriptions in results.items():
    for model_name, text in transcriptions.items():
        is_error = text.startswith("[ERROR]")
        rows.append({
            "strip": sid,
            "model": model_name,
            "chars": len(text) if not is_error else 0,
            "words": len(text.split()) if not is_error else 0,
            "lines": text.count("\n") + 1 if not is_error else 0,
            "error": is_error,
        })

df = pd.DataFrame(rows)

print("=" * 60)
print("  Per-strip character counts")
print("=" * 60)
pivot = df.pivot_table(index="strip", columns="model", values="chars", aggfunc="first")
display(pivot)

print("\n" + "=" * 60)
print("  Aggregate stats per model")
print("=" * 60)
agg = df.groupby("model").agg(
    total_chars=("chars", "sum"),
    total_words=("words", "sum"),
    avg_chars=("chars", "mean"),
    errors=("error", "sum"),
).sort_values("total_chars", ascending=False)
display(agg)

In [ ]:
# Bar chart: characters per model per strip
if not df.empty and not df["chars"].eq(0).all():
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(kind="bar", ax=ax, edgecolor="white", linewidth=0.5)
    ax.set_ylabel("Characters")
    ax.set_title("Transcription length by model and strip")
    ax.legend(title="Model", fontsize=7, title_fontsize=8)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()